In [3]:
## Removing PCA completely and use Ridge regression

In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, GridSearchCV, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error


# ============================================================
# 1. Load data
# ============================================================

df = pd.read_csv("wide_diff_all_data.csv")

print("Number of subjects:", len(df))


# ============================================================
# 2. Select diffusion features
# ============================================================
diffusion_substrings = [
    "dki_fa",
    "dki_md",
    "dki_mk",
    "dki_awf"
]

diffusion_cols = [
    col for col in df.columns
    if any(x in col for x in diffusion_substrings)
]

print("Number of diffusion features:", len(diffusion_cols))

# ============================================================
# 3. Prepare target
# ============================================================
y = pd.to_numeric(df["DDisc_AUC_40K"],    errors="coerce")

# ============================================================
# 4. Prepare covariates
# ============================================================
covariates = df[
    [
        "age_group",
        "Gender"
    ]
].copy()

# Convert age and sex to numeric variables
covariates = pd.get_dummies(
    covariates,
    columns=[
        "age_group",
        "Gender"
    ],
    drop_first=True
)


# ============================================================
# 5. Create final feature matrix
# ============================================================

X_diff = df[diffusion_cols].copy()

# Ensure numeric
X_diff = X_diff.apply(pd.to_numeric, errors="coerce")


# Combine diffusion + covariates
X = pd.concat(
    [
        X_diff,
        covariates
    ],
    axis=1
)


# Remove missing subjects
data = pd.concat([ X, y], axis=1).dropna()

X = data.drop( columns=["DDisc_AUC_40K"])
y = data["DDisc_AUC_40K"].astype(float)

print("Final subjects:", len(y))
print("Final features:", X.shape[1])


# ============================================================
# 6. Nested cross-validation
# ============================================================
outer_cv = KFold(n_splits=5, shuffle=True, random_state=42)
inner_cv = KFold(n_splits=5, shuffle=True, random_state=42)

# ============================================================
# 7. Ridge pipeline
# ============================================================
pipeline = Pipeline(    [
        (
            "scaler",
            StandardScaler()
        ),

        (
            "model",
            Ridge()
        )
    ]
)

# Ridge regularization strength
param_grid = {
    "model__alpha":
        np.logspace(-3, 3, 50 )
}


search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=inner_cv,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)


# ============================================================
# 8. Nested CV prediction
# ============================================================
y_pred = cross_val_predict(search,X,y,cv=outer_cv,n_jobs=-1)

# ============================================================
# 9. Evaluation
# ============================================================
r2 = r2_score(y,y_pred)

rmse = np.sqrt(mean_squared_error(y,y_pred))
mae = mean_absolute_error(y,y_pred)

print("==============================")
print("Nested CV Ridge Results")
print("==============================")

print(f"R²   : {r2:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")


# ============================================================
# 10. Best Ridge parameter
# ============================================================

search.fit(X,y)

print("\nBest parameter:")
print(search.best_params_)

Number of subjects: 691
Number of diffusion features: 96
Final subjects: 691
Final features: 100
Nested CV Ridge Results
R²   : -0.0189
RMSE : 0.2862
MAE  : 0.2464

Best parameter:
{'model__alpha': 1000.0}


### Random Forest without PCA and RandomizedSearchCV

In [1]:
import numpy as np
import pandas as pd

from scipy.stats import randint

from sklearn.model_selection import (
    KFold,
    RandomizedSearchCV,
    cross_val_predict
)

from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

# ============================================================
# 1. Load dataset
# ============================================================

df = pd.read_csv("wide_diff_all_data.csv")

print("Number of subjects:", len(df))

# ============================================================
# 2. Define diffusion features
# ============================================================

diffusion_substrings = [
    "dki_fa",
    "dki_md",
    "dki_mk",
    "dki_awf"
]

diffusion_cols = [
    col for col in df.columns
    if any(s in col for s in diffusion_substrings)
]

print("Number of diffusion features:", len(diffusion_cols))

# ============================================================
# 3. Outcome
# ============================================================

y = pd.to_numeric(
    df["DDisc_AUC_40K"],
    errors="coerce"
)

# ============================================================
# 4. Covariates
# ============================================================

covariates = df[
    [
        "age_group",
        "Gender"
    ]
].copy()

covariates = pd.get_dummies(
    covariates,
    columns=[
        "age_group",
        "Gender"
    ],
    drop_first=True
)

# ============================================================
# 5. Feature matrix
# ============================================================

X_diff = df[diffusion_cols].apply(
    pd.to_numeric,
    errors="coerce"
)

X = pd.concat(
    [
        X_diff,
        covariates
    ],
    axis=1
)

complete = pd.concat(
    [
        X,
        y
    ],
    axis=1
).dropna()

X = complete.drop(columns=["DDisc_AUC_40K"])
y = complete["DDisc_AUC_40K"].astype(float)

print("Final subjects:", len(y))
print("Final features:", X.shape[1])

# ============================================================
# 6. Cross-validation
# ============================================================

outer_cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

inner_cv = KFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

# ============================================================
# 7. Random Forest
# ============================================================

pipeline = Pipeline([
    (
        "model",
        RandomForestRegressor(
            random_state=42,
            n_jobs=-1
        )
    )
])

# ============================================================
# 8. Hyperparameter distributions
# ============================================================

param_dist = {

    "model__n_estimators":
        randint(100, 301),

    "model__max_depth":
        [
            None,
            5,
            10,
            20
        ],

    "model__min_samples_split":
        randint(2, 10),

    "model__min_samples_leaf":
        randint(1, 5),

    "model__max_features":
        [
            "sqrt",
            0.3,
            0.5
        ]
}

search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_dist,
    n_iter=30,
    cv=inner_cv,
    scoring="neg_mean_squared_error",
    random_state=42,
    n_jobs=-1
)

# ============================================================
# 9. Nested Cross Validation
# ============================================================

print("\nRunning nested cross-validation...")

y_pred = cross_val_predict(
    search,
    X,
    y,
    cv=outer_cv,
    n_jobs=-1
)

# ============================================================
# 10. Performance
# ============================================================

rmse = np.sqrt(
    mean_squared_error(y, y_pred)
)

mae = mean_absolute_error(
    y,
    y_pred
)

r2 = r2_score(
    y,
    y_pred
)

print("\n==============================")
print("Nested CV Random Forest")
print("==============================")

print(f"R²   : {r2:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")

# ============================================================
# 11. Best model (optional)
# ============================================================

print("\nFinding best parameters on full dataset...")

search.fit(X, y)

print("\nBest parameters:")
print(search.best_params_)

Number of subjects: 691
Number of diffusion features: 96
Final subjects: 691
Final features: 100

Running nested cross-validation...

Nested CV Random Forest
R²   : -0.0141
RMSE : 0.2855
MAE  : 0.2469

Finding best parameters on full dataset...

Best parameters:
{'model__max_depth': None, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 3, 'model__min_samples_split': 3, 'model__n_estimators': 231}
